In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
def categorize_labels(label: float) -> float:
    if label < 0.1:
        return 0.0
    
    return 1.0

In [ ]:
data = np.loadtxt('stock_context_curated.csv', delimiter=',')
data = data.T
np.random.shuffle(data)
data = data.T

In [ ]:
raw_T = data[0:252,:]
raw_SP = data[252:504,:]
raw_Y = np.array(data[-1:,:])
X = np.array(list(zip(raw_T, raw_SP))) # Zip to make s&p and ticker days align
Y = (tf.keras.utils.to_categorical(raw_Y, 2)).T

assert(raw_T[0,0] == X[0,0,0] and raw_SP[0,0] == X[0,1,0])

In [ ]:
Y.shape = (Y.shape[0], Y.shape[1])
print(X.shape)
print(Y.shape)

In [ ]:
print("Worse Than s&p:", np.count_nonzero(Y[0]))
print("Better Than s&p:", np.count_nonzero(Y[1]))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(2, 252)),
    tf.keras.layers.Dense(50, activation='tanh', kernel_initializer='glorot_uniform'),
    tf.keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(50, activation='tanh', kernel_initializer='glorot_uniform'),
    tf.keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'),
    #tf.keras.layers.Reshape((252,1)),
    #tf.keras.layers.LSTM(252, activation='tanh', recurrent_activation='relu', kernel_initializer='he_normal'),
    #tf.keras.layers.Reshape((512, 1)),
    #tf.keras.layers.Conv1D(filters=30, kernel_size=7, activation='relu', padding='same'),
    #tf.keras.layers.Reshape((14,1)),
    #tf.keras.layers.MaxPooling1D(pool_size=2),
    #tf.keras.layers.Flatten(), 
    tf.keras.layers.Dense(2, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
train_size = int(data.shape[1] * 0.8)

X_train = X[:,:,:train_size].T
Y_train = Y[:,:train_size].T

X_val = X[:,:,train_size:].T
Y_val = Y[:,train_size:].T

mean = X_train.mean(axis=0)
std  = X_train.std(axis=0) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std

print(X_train.shape)
print(Y_train.shape)
print(X_val.shape)
print(Y_val.shape)

In [ ]:
model.fit(X_train, Y_train,
                  validation_data=(X_val,Y_val),
                  batch_size=64, 
                  epochs=300,
                  verbose=1,
                  shuffle=True)

In [ ]:
import matplotlib.pyplot as plt

def plot_acc_and_loss(history, title):
    plt.title("Model and Validation MAE for " + title)
    xs = list(range(1, len(history.history['accuracy']) + 1))
    plt.plot(xs, history.history['accuracy'], label="Model Accuracy", color="Red")
    plt.plot(xs, history.history['val_accuracy'], label="Validation Accuracy", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    plt.show()

In [ ]:
plot_acc_and_loss(model.history, "Test")

In [ ]:
print("Prediction:", model(X_val[5:10]))
print("Label:", Y_val[5:10])